In [1]:
import numpy as np
from cost_function import sigmoid, ReLU, I

In [ ]:
# Neural Network
# I want to make 3 layer NN that consists of 3-3-2 nodes each.

X = np.array([1,-1])

W1 = np.array([[0.2, 0.3, 0.5],[0.4, 0.2, 0.1]])
W2 = np.array([[0.2, 0.4, 0.1],[0.2, 0.2, 0.3], [0.4, 0.2, 0.3]])
W3 = np.array([[0.2, 0.3],[0.7, 0.2],[0.3, 0.1]])

B1 = np.array([-0.4,0.7,0.2])
B2 = np.array([0.4,-0.7,-0.2])
B3 = np.array([-0.2,0.2])


def forward(x, w1,w2,w3,b1,b2,b3):
    x = np.dot(x, W1) + b1
    x = ReLU(x)
    x = np.dot(x, W2) + b2
    x = ReLU(x)
    x = np.dot(x, W3) + b3
    x = I(x)
    return x


res = forward(X, W1,W2,W3,B1,B2,B3)
print(f"result: {res}")

result: [0.026 0.462]


In [ ]:
import numpy as np
import copy

from cost_function import sigmoid, CrossEntropyLoss

def numerical_diff(f, x):
    '''
    The function that gets the gradient value of vector-valued funciton f at x

    f : A vector-valued function f
    x : A point x 
    '''
    tmp_x = copy.deepcopy(x)
    grad = np.zeros_like(x)
    h = 1e-4

    for i in range(x.size):
        hej = np.zeros_like(x) 
        hej[i] = h

        fx1 = f(tmp_x + hej) 
        fx2 = f(tmp_x - hej)

        grad[i] = (fx1-fx2)/(2*h)

    return grad

class TwoLayerNet():         # 784 -> 50 - 50 -> 10
    def __init__(self, input_size, output_size, hidden_size, batch_size):
        self.W = {}
        self.W['W1'] = np.random.randn(input_size, hidden_size)
        self.W['b1'] = np.zeros((batch_size, hidden_size))
        self.W['W2'] = np.random.randn(hidden_size, output_size)
        self.W['b2'] = np.zeros((batch_size, hidden_size))


    def loss(self, x, t):
        y = self.predict(x)
        return CrossEntropyLoss(t, y)


    def accuracy(self, x, t):
        y = self.predict(x)
        pred = np.argmax(y)

        cnt = np.sum(pred==t)
        return cnt/pred.shape[0]


    def predict(self, x):
        """
        A method that predict the vector consisting of the probabilities of outputs
        """
        z1 = np.dot(x, self.W['W1']) + self.W['b1']
        o1 = sigmoid(z1)
        z2 = np.dot(o1, W2) + self.W['b2']
        o2 = I(z2)

        return o2
    

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads
        

batch_size = 100
iteration = 10000
lr = 1e-4
train_losses = []

x = np.random.rand(10000, 784)
t = np.random.rand(10000, 10)
net = TwoLayerNet(784, 10, 50, batch_size)


for i in range(iteration):
    batch_ind = np.random.choice(x.shape[0], batch_size)
    x_batch = x[batch_ind]
    t_batch = t[batch_ind]

    grad = net.numerical_gradient(x_batch, t_batch)

    for key in ('W1', 'b1', 'W2', 'b2'):
        net.W[key] -= lr * grad[key]

    loss = net.loss(x_batch, t_batch)
    train_losses.append(loss)



In [ ]:
class TwoLayerNN:

    def __init__(self, input_size, output_size, hidden_size, weight_init_std=0.01):
        rn = np.random.randn
        zr = np.zeros

        self.params = []
        self.params.append(weight_init_std * rn(input_size, hidden_size))
        self.params.append(rn(hidden_size, output_size))
        self.params.append(weight_init_std * zr((1, hidden_size)))
        self.params.append(zr((1, output_size)))

        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['ReLU'] = ReLU()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])
        self.last_layer = SoftmaxWithLoss() 

        self.Loss = None
        self.grads = None
    

    def predict(self, X):
        for layer in self.layers.values():
            X = layer.forward(X)
        return X

    def loss(self, X, tgt):
        pred = self.predict(X)
        loss = self.last_layer.forward(tgt, pred)
        return loss
    
    def accuracy(self, X, tgt):
        pred = self.predict(X)
        pred = np.argmax(pred, axis=1)
        tgt = np.argmax(tgt, axis=1)

        acc = np.sum(pred == tgt) / float(X.shape[0])
        return acc

    
    def forward(self, X, tgt):
        for layer in self.layers.values():
            X = layer.forward(X)
        loss = self.last_layer.forward(tgt, X)

        self.Loss = loss
        return loss


    def backward(self, dout=1):
        dout = self.last_layer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()

        for layer in layers:
            dout = layer.backward(dout)

        grads = []
        grads.append(self.layer['Affine1'].dW)
        grads.append(self.layer['Affine2'].dW)
        grads.append(self.layer['Affine1'].db)
        grads.append(self.layer['Affine2'].db)

        self.grads = grads
        return grads